# Results for model: qwen/qwen2.5-coder-32b-instruct

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Load data
data = pl.read_parquet('./jane_street_data/train.parquet')

# Feature Engineering
TOP_QUANTILE = 0.85  # Top 15% is the bottom 85% quantile

# Calculate rolling quantile of 'feature_00' for 'responder_6' == 1
quantiles = data.filter(pl.col('responder_6') == 1).groupby('date_id').agg(
    pl.col('feature_00').quantile(TOP_QUANTILE).alias('feature_00_quantile')
)

# Join quantiles back to the original data
data = data.join(quantiles, on='date_id', how='left')

# Create a new feature indicating if 'feature_00' is in the top quantile
data = data.with_column((pl.col('feature_00') >= pl.col('feature_00_quantile')).alias('feature_00_in_top_quantile'))

# Prepare data for modeling
data = data.drop_nulls()
X = data[[col for col in data.columns if col.startswith('feature_')] + ['feature_00_in_top_quantile']]
y = data['responder_6']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X.to_pandas(), y.to_pandas(), test_size=0.2, random_state=42)

# Train XGBoost Regressor
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'seed': 42
}

bst = xgb.train(params, dtrain, num_boost_round=100, evals=[(dtest, "eval")], early_stopping_rounds=10)

# Evaluate model
preds = bst.predict(dtest)
rmse = mean_squared_error(y_test, preds, squared=False)
print(f'RMSE: {rmse}')